In [1]:
import torch
from torch import nn

# 对于L1Loss数据类型dtype要求是torch.float32或者torch.float
inputs = torch.tensor([1, 2, 3], dtype=torch.float32)
targets = torch.tensor([1, 2, 5], dtype=torch.float32)

inputs = torch.reshape(inputs, (1, 1, 1, 3))
targets = torch.reshape(targets, (1, 1, 1, 3))

print(inputs.shape)
print(targets.shape)
print(inputs)
print(targets)


torch.Size([1, 1, 1, 3])
torch.Size([1, 1, 1, 3])
tensor([[[[1., 2., 3.]]]])
tensor([[[[1., 2., 5.]]]])


In [ ]:

Loss = nn.L1Loss()
result = Loss(inputs, targets)
# L1Loss的计算方式是：每个元素的绝对值差的平均值
print(result)
# L1Loss = 1/3 * (|1-1| + |2-2| + |3-5|) = 1/3 * 2 = 0.6667

tensor(0.6667)


In [ ]:
Loss_mse = nn.MSELoss()
result_mse = Loss_mse(inputs, targets)
# MSELoss的计算方式是：每个元素的平方差的平均值
print(result_mse)
# MSELoss = 1/3 * ((1-1)^2 + (2-2)^2 + (3-5)^2) = 1/3 * 4 = 1.3333

tensor(1.3333)


Optimized tool selection**交叉熵损失（Cross-Entropy Loss）**是深度学习中分类任务最常用的损失函数。它主要用来衡量“模型预测的概率分布”和“真实值的概率分布”之间的差异。差异越大，交叉熵损失就越大；差异越小，损失就越接近于 0。

### 1. 核心概念
在多分类任务中，模型最后一层通常会输出每个类别的得分（Logits）。交叉熵损失的计算目的，就是**把正确类别的得分拉高，把其他错误类别的得分压低**。

### 2. PyTorch 中的 `nn.CrossEntropyLoss()`
在 PyTorch 中，`nn.CrossEntropyLoss()` 内部实际上结合了两个操作：
1. **`LogSoftmax`**：将网络输出的原始得分（Logits）转化为概率分布（所有类别的概率和为1），再取对数。
2. **`NLLLoss`（负对数似然损失）**：在使用 `LogSoftmax` 处理后的结果中，提取出真实标签对应位置的值，并取负号作为最终的 Loss。

**⚠️ 重要注意：**
因为 `nn.CrossEntropyLoss()` 内部已经包含了 Softmax 操作，所以**神经网络的最后一层不需要也不应该加 Softmax 激活函数**，直接输出 raw scores（原始得分）即可。

### 3. 代码示例
对于你的 `P23_loss.ipynb`，你可以写一个简单的例子来进行测试：
```python
import torch
from torch import nn

# 假设 batch_size=1，有 3 个类别 (例如: 狗, 猫, 猪)
# inputs 是模型输出的未经过 softmax 的原始得分 (Logits)
x = torch.tensor([[0.2, 0.1, 0.9]]) 

# targets 是真实的标签索引 (假设真实类别是索引为 2 的那类)
y = torch.tensor([2]) 

loss_cross = nn.CrossEntropyLoss()
result_cross = loss_cross(x, y)

print(result_cross)
```

**应用场景**：大部分的图像分类任务（如猫狗分类、CIFAR-10 等）在计算最终误差时，使用的都是交叉熵损失函数。

交叉熵（Cross-Entropy）公式主要用于衡量真实的概率分布与模型预测的概率分布之间的差异。

### 1. 标准公式
在多分类问题中，交叉熵损失的计算公式如下：

$$ Loss = -\sum_{i=1}^{C} y_i \log(\hat{y}_i) $$

**公式解释：**
* $C$：表示类别的总数。
* $y_i$：表示**真实标签**。通常是一个 One-Hot 编码的向量（即真实类别的那个位置是 1，其他位置都是 0）。
* $\hat{y}_i$：表示**模型预测**属于第 $i$ 个类别的**概率**（通常是经过 Softmax 激活函数后的输出，介于 0 和 1 之间）。

### 2. 公式的简化（重中之重）
因为真实的标签 $y$ 是 One-Hot 编码（例如类别总数为 3，真实标签是第 0 类，那么 $y = [1, 0, 0]$），所以求和符号 $\sum$ 里的绝大多数项都会变成 0。

整个公式最终会坍缩为只计算**正确类别的对数损失**：

$$ Loss = -\log(\hat{y}_{c}) $$

*其中 $c$ 表示正确的那个类别的索引。*

**通俗理解：**
* 这个公式意味着：**交叉熵损失只关心“模型对正确分类预测的概率（$\hat{y}_c$）”**。
* 如果模型对正确选项预测的概率非常高（$\hat{y}_c \to 1$），那么 $\log(1) = 0$，损失就接近 0。
* 如果模型对正确选项预测的概率非常低（$\hat{y}_c \to 0$），那么 $\log(0) \to -\infty$，加上前面的负号，损失就会趋近于正无穷大。

### 3. 在 PyTorch 中的实际公式 (`nn.CrossEntropyLoss`)
在 PyTorch 中，`nn.CrossEntropyLoss` 接收的是网络输出的原始得分（Logits，用 $x$ 表示）和一个真实的类别索引（用 $class$ 表示），其实际计算公式是将 Softmax 和上述公式结合在了一起：

$$ \text{Loss}(x, class) = -\log\left(\frac{\exp(x[class])}{\sum_{j} \exp(x[j])}\right) = -x[class] + \log\left(\sum_{j} \exp(x[j])\right) $$

* $x[class]$：模型输出的属于**正确类别**的得分。
* 分母部分是对所有类别的得分求指数和，用来计算概率（Softmax 的过程）。
* 也就是“先算 Softmax 得到预测概率，再对其取对数，最后乘以 -1”。

log应该是ln


完全正确！在深度学习和 PyTorch 的官方文档中，**公式里写到的 $\log$ 实际上指的都是自然对数 $\ln$（以 $e$ 为底数）**。

因此，更严谨的表达方式如下：

### 1. 标准交叉熵公式
$$ Loss = -\sum_{i=1}^{C} y_i \ln(\hat{y}_i) $$
简化后，只看正确类别的预测概率 $\hat{y}_c$：
$$ Loss = -\ln(\hat{y}_{c}) $$

### 2. Pytorch 中的 `nn.CrossEntropyLoss` 公式
因为 Softmax 操作是以 $e$ 为底数的指数操作（$e^x$），所以当我们对它取自然对数 $\ln$ 时，很多项可以完美抵消，起到简化计算和防止数值溢出的作用（这就是所谓的 `LogSoftmax`）：

$$ \text{Loss}(x, class) = -\ln\left(\frac{e^{x[class]}}{\sum_{j} e^{x[j]}}\right) = -x[class] + \ln\left(\sum_{j} e^{x[j]}\right) $$

**总结：**
在绝大多数机器学习的论文和代码库（包括 PyTorch 的底层实现）中，如果没有特殊说明，**$\log$ 默认就是自然对数 $\ln$**。非常感谢指出，你的指正是非常严谨的！

In [7]:
#cross entropy loss输入概率分布和目标类别标签
#输入的概率分布应该是经过softmax处理的，而目标类别标签应该是整数形式的类别索引。
#注意前面的说明，nn.CrossEntropyloss()包含了softmax函数，所以输入的概率分布不需要经过softmax处理，直接输入原始的logits即可。


x = torch.tensor([0.1, 0.2, 0.3])
y = torch.tensor([1])   # y的值是1，表示正确类别的索引是1，对应的元素为x中的索引1即第2个元素0.2
x = torch.reshape(x, (1, 3))
loss_cross_entropy = nn.CrossEntropyLoss()
result_cross_entropy = loss_cross_entropy(x, y)
# CrossEntropyLoss的计算方式是：-x[class] + ln(sum(exp(x[j])))，其中class是正确类别的索引
print(result_cross_entropy)
# CrossEntropyLoss = -0.2 + ln(exp(0.1) + exp(0.2) + exp(0.3)) = 1.1019


tensor(1.1019)


###  loss_network

In [14]:


from torch import nn
import torchvision

class seq(nn.Module):
    def __init__(self):
        super(seq, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=5, padding=2, stride=1)
        # 假设输入图像尺寸为32*32，输入通道数为3（RGB图像），则按以上设置卷积核大小为5*5，输出通道数为32，步长为1，填充为2，
        # 得到输出尺寸不变，仍为32*32，
        # 计算公式：output_size = (input_size - kernel_size + 2 * padding) / stride + 1
        # 注意区分in_channels和input_size，in_channels是输入的通道数，input_size是输入的空间尺寸（宽和高）
        # 32 = (32 - 5 + 2 * padding) / 1 + 1
        # padding=2，kernel_size=5，stride=1满足输出尺寸不变的条件：padding = (kernel_size - 1) / 2

        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, padding=2, stride=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, padding=2, stride=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        # flatten将输入的64张4*4的特征图展平为一维向量，长度为64*4*4=1024
        # 1024 = 64 * 4 * 4
        self.linear1 = nn.Linear(1024, 64)
        # 所谓线性层，就是全连接层，输入的每个元素都与输出的每个元素相连，输出的每个元素都是输入的所有元素的线性组合
        self.linear2 = nn.Linear(64, 10)


        self.model1 = nn.Sequential(
            self.conv1,
            self.pool1,
            self.conv2,
            self.pool2,
            self.conv3,
            self.pool3,
            self.flatten,
            self.linear1,
            self.linear2
        )


    def forward(self, input):
        # output = self.conv1(input)
        # output = self.pool1(output)
        # output = self.conv2(output)
        # output = self.pool2(output)
        # output = self.conv3(output)
        # output = self.pool3(output)
        # output = self.flatten(output)
        # output = self.linear1(output)
        # output = self.linear2(output)
        output = self.model1(input)

        return output


dataset = torchvision.datasets.CIFAR10(root="./dataset", train=True, transform=torchvision.transforms.ToTensor(), download=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True)

net = seq()
loss = nn.CrossEntropyLoss()
for data in dataloader:
    inputs, targets = data
    outputs = net(inputs)
    result_loss = loss(outputs, targets)
    # print(outputs)
    # print(targets)
    print(result_loss)


Files already downloaded and verified
tensor(2.3303, grad_fn=<NllLossBackward0>)
tensor(2.3221, grad_fn=<NllLossBackward0>)
tensor(2.2941, grad_fn=<NllLossBackward0>)
tensor(2.3253, grad_fn=<NllLossBackward0>)
tensor(2.3113, grad_fn=<NllLossBackward0>)
tensor(2.3068, grad_fn=<NllLossBackward0>)
tensor(2.3855, grad_fn=<NllLossBackward0>)
tensor(2.3142, grad_fn=<NllLossBackward0>)
tensor(2.2783, grad_fn=<NllLossBackward0>)
tensor(2.3136, grad_fn=<NllLossBackward0>)
tensor(2.3223, grad_fn=<NllLossBackward0>)
tensor(2.3355, grad_fn=<NllLossBackward0>)
tensor(2.3741, grad_fn=<NllLossBackward0>)
tensor(2.2933, grad_fn=<NllLossBackward0>)
tensor(2.3325, grad_fn=<NllLossBackward0>)
tensor(2.3245, grad_fn=<NllLossBackward0>)
tensor(2.3101, grad_fn=<NllLossBackward0>)
tensor(2.3045, grad_fn=<NllLossBackward0>)
tensor(2.3438, grad_fn=<NllLossBackward0>)
tensor(2.2926, grad_fn=<NllLossBackward0>)
tensor(2.2578, grad_fn=<NllLossBackward0>)
tensor(2.2854, grad_fn=<NllLossBackward0>)
tensor(2.3741, g